# Model Comparison

This notebook benchmarks multiple machine learning algorithms for the credit card fraud detection task.

The goal is to identify the most promising model to be used in the next stage of the project, where hyperparameter optimization will be performed.

## Objectives

- Train baseline models using the feature-engineered dataset
- Evaluate model performance using metrics suitable for imbalanced classification
- Compare models using a consistent evaluation framework
- Select the best-performing model for further optimization

## Configuration

All project configurations are centralized in the configuration.yaml file located in the project root directory.
This file contains parameters related to:

- Data paths
- Model configurations
- Experiment settings

The notebook loads these parameters to ensure reproducibility and consistency across experiments.

## Data Source

This notebook depends on the output generated by the notebook:

    3-Feature-Engineering.ipynb

If that notebook has not been executed previously, the required dataset will not exist.

The training dataset used in this stage is located at:

    ../data/splits/feature_engineered/train_fe.parquet

This path is relative to the notebook location within the project structure.

## Models Evaluated

The following algorithms are evaluated:

- Logistic Regression
- Random Forest
- Extra Trees
- AdaBoost
- XGBoost
- LightGBM
- CatBoost
- Multi-Layer Perceptron (MLP)

Each model is trained using baseline parameters to establish a performance benchmark.

## Model Selection

Models are compared using evaluation metrics appropriate for fraud detection, where class imbalance is significant.
The best-performing model will be selected for the next stage of the project:

    Hyperparameter Optimization.

## Research vs Production Code

This notebook was created during the research and experimentation phase of the project.
While it contains exploratory implementations, the production-ready data preparation pipeline is implemented in:

        src/datapipeline
        
This ensures that the final workflow used in production is modular, testable, and reproducible.

In [ ]:
from datapipeline.training.load_data import load_raw_data
from datapipeline.config.mlflow_config import setup_mlflow
import yaml
import pandas as pd
import mlflow
from pathlib import Path
import tempfile
from sklearn.metrics import (roc_auc_score, 
                            balanced_accuracy_score,
                            precision_score,
                            recall_score,
                            f1_score,
                            confusion_matrix,
                            precision_recall_curve,
                            average_precision_score,
                            make_scorer)
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight

# Configuration

In [ ]:
#primary metric that will be used for model comparison
PRIMARY_METRIC = "Average Precision Score"

In [ ]:
config_file_path = '../config.yaml'


In [ ]:
#There is a config.yaml file at the project root
#loading config file
config_path = config_file_path
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
    

# MLflow

In [ ]:
mlflow_local_folder = config['mlflow']['experiment_path']

In [ ]:
experiment_name = config['mlflow']['experiment_name']

In [ ]:
setup_mlflow(experiment_name, mlflow_local_folder)

In [ ]:
mlflow.start_run(run_name='Model Selection')

In [ ]:
mlflow.set_tag('Type', 'Research')
mlflow.set_tag('Notebook', '4-Model_Comparisson')

In [ ]:
artifacts_path_mlflow = config['4-model_selection']['artifacts_path']

# Load Dataset


In [ ]:
dataset_path = Path(config['datasets']['train_feature_engineered_path'])
target_column = config['2-data_cleaning_and_split']['target_column']

In [ ]:
df_train = pd.read_parquet(dataset_path)

In [ ]:
y_train = df_train[target_column]

In [ ]:
x_train = df_train.drop(columns = ['Time', target_column])

# Models

In [ ]:
random_state = config['parameters']['random_state']

In [ ]:
#models that will be tested
dummy = DummyClassifier(strategy = 'most_frequent')
lr = LogisticRegression(max_iter=1000,
                       random_state=random_state, verbose=False)
catboost = CatBoostClassifier(iterations=100, random_state=random_state, verbose=False)
xgboost = xgb.XGBClassifier(n_estimators=100, random_state=random_state)
adaboost = AdaBoostClassifier(n_estimators=100, random_state=random_state)
rf = RandomForestClassifier(n_estimators=100, random_state=random_state, verbose=False)
extra_tree = ExtraTreesClassifier(n_estimators=100 , random_state=random_state, verbose=False)
lgb = LGBMClassifier(n_estimators=100, random_state=random_state)
mlp = MLPClassifier(random_state=random_state, verbose=False)


In [ ]:
candidate_models = {'Dummy': dummy,
                    'Logistic Regression': lr,
                   'CatBoost': catboost,
                   'XgBoost': xgboost,
                   'AdaBoost': adaboost,
                   'Random Forest': rf,
                   'Extra Trees': extra_tree,
                   'LightGBM': lgb, 
                   'MLP': mlp}

In [ ]:
#metrics that will be used for model comparison
metrics = ['Balanced Accuracy Score',
           'Precision Score',
           'Recall Score',
           'F1 Score',
           'Average Precision Score',
           'Roc AUC']


In [ ]:
scoring = {
    'balanced_accuracy': 'balanced_accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'average_precision': 'average_precision',
    'roc_auc': 'roc_auc'
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

# First Tests 

In [ ]:
results_first_tests = pd.DataFrame(columns = metrics,
                       index= candidate_models.keys(),
                       data = 0.0)
results_first_tests

## All selected models will be trained with their default parameter settings, except for the number of estimators used in the ensemble models.

In [ ]:
for model_name, model in candidate_models.items():
    print(f'Training model {model_name}')

    cv_results = cross_validate(model, 
                                x_train, 
                                y_train, 
                                cv=cv, 
                                scoring=scoring, 
                                verbose = False)
   
    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    print(metrics_results)
    results_first_tests.loc[model_name, :] = metrics_results


In [ ]:
results_first_tests = results_first_tests.rename_axis('Models')


In [ ]:
results_first_tests =results_first_tests.sort_values(by = 'Average Precision Score', 
                                                     ascending = False)
results_first_tests

In [ ]:
def log_artifacts(dataframe: pd.DataFrame, 
                  file_name: str, 
                  artifacts_path_mlflow: str) -> None:
    
    '''
    Log a dataframe to mlflow as an artifact and as a table.

    Args:
        dataframe (pd.DatsFrame): The pandas dataframe to be logged.
        file_name (str): name of the files
        artifacts_path_mlflow (str): folder to save the artifacts in mlflow
    
    '''

    df_log = dataframe.reset_index()

    with tempfile.TemporaryDirectory() as tmp_dir:
        file_path_parquet = Path(tmp_dir) / f"{file_name}.parquet"
        file_path_json = Path(tmp_dir) / f"{file_name}.json"

        # save parquet
        df_log.to_parquet(file_path_parquet)

        #save json
        #df_log.to_json(file_path_json, orient="split")
        
        
        # log dataset completo
        mlflow.log_artifact(
            str(file_path_parquet),
            artifact_path=artifacts_path_mlflow
        )

        # log preview visualizável no MLflow
        mlflow.log_table(
            df_log,
            artifact_file=f"{artifacts_path_mlflow}/{file_name}.json"
        )

In [ ]:
log_artifacts(results_first_tests, 'first_tests'
              , artifacts_path_mlflow)

# Second Tests

## Adjusting a hyperparameter in certain models to address the class imbalance in the dataset.

In [ ]:
candidate_models_second_tests = [
                   'Dummy',
                   'Logistic Regression',
                   'CatBoost',
                   'XgBoost',
                   'AdaBoost',
                   'Random Forest',
                   'Extra Trees',
                   'LightGBM']

In [ ]:
results_second_tests = pd.DataFrame(columns = metrics,
                       index= candidate_models_second_tests,
                       data = 0.0)
results_second_tests

In [ ]:
for model_name in candidate_models_second_tests:

    if model_name == 'Dummy':
        model_adjusted = DummyClassifier(strategy = 'most_frequent')
    
    elif model_name == 'Logistic Regression':
        model_adjusted = LogisticRegression(max_iter=5000,
                                           random_state=random_state,
                                           class_weight='balanced', verbose=False)        
    
    elif model_name == 'CatBoost':
        model_adjusted = CatBoostClassifier(iterations=100,
                                            auto_class_weights='Balanced',
                                            random_state=random_state, verbose=False)

    elif model_name == 'XgBoost':
        n_major = sum(y_train==0)
        n_minor = sum(y_train==1)
        scale_pos_weight = n_major / n_minor

        model_adjusted = xgb.XGBClassifier(n_estimators=100, 
                                           random_state=random_state,
                                           scale_pos_weight=scale_pos_weight)
    elif model_name == 'AdaBoost':
        base_tree = DecisionTreeClassifier(
                    max_depth=1,
                    class_weight='balanced',
                    random_state=random_state)
        model_adjusted = AdaBoostClassifier(n_estimators=100, 
                                      estimator=base_tree,
                                      random_state=random_state)

    elif model_name == 'Random Forest':
        model_adjusted = RandomForestClassifier(n_estimators=100, 
                                    class_weight='balanced',
                                    random_state=random_state,
                                    verbose=False)
                                            
    elif model_name == 'Extra Trees':
        model_adjusted = ExtraTreesClassifier(n_estimators=100,
                                          class_weight='balanced',
                                          random_state=random_state,
                                          verbose=False)                           
    elif model_name == 'LightGBM':
        model_adjusted = LGBMClassifier(n_estimators=100, 
                        random_state=random_state,
                        class_weight='balanced')

   
                                              
    
    print(f'Training model {model_name}')
    cv_results = cross_validate(model_adjusted, x_train, y_train, cv=cv, scoring=scoring, verbose=False)
    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    print(metrics_results)
    results_second_tests.loc[model_name, :] = metrics_results
    

In [ ]:
results_second_tests = results_second_tests.sort_values(by=PRIMARY_METRIC, ascending=False) 
results_second_tests

In [ ]:
results_second_tests = results_second_tests.rename_axis('Models')


In [ ]:
log_artifacts(results_second_tests,
              'results_second_tests',
               artifacts_path_mlflow)

# Comparison

## Combining the results of both tests into a single DataFrame

In [ ]:
comparison = pd.concat(
    [results_first_tests, results_second_tests
    ],
    axis=1,
    join='inner',
    keys=['First Tests', 'Second Tests']
)

In [ ]:
comparison.columns.names = ['Tests', 'Metrics']


In [ ]:
comparison = comparison.sort_values(by=[('First Tests','Average Precision Score')], ascending=False)
comparison

In [ ]:
log_artifacts(comparison,
               'comparison',
               artifacts_path_mlflow
             )

## Percentage difference between the second and first tests

In [ ]:
diff_pct = ((comparison['Second Tests'] - comparison['First Tests'])/ comparison['First Tests'])*100

In [ ]:
diff_pct.columns = [f'{column_name} (%)' for column_name in diff_pct.columns]

In [ ]:
diff_pct.style.format('{:.2f}')

In [ ]:
log_artifacts(diff_pct,
             'percentage_difference',
             artifacts_path_mlflow)

In [ ]:
text = """
##First Tests
All Models were trained using their default parameters

## Second Tests
A parameter in each model was adjusted to address class imbalance. 
It may have different names depending on the model, such as scale_pos_weight in 
the XGBoost model and class_weight in the AdaBoost model. Despite the different names, 
the underlying idea is the same: mistakes are penalized according to the class of the sample.

## Comparison
It shows the percentage difference between the first and second tests.
"""

mlflow.log_text(text, "documentation/tables_description.md")

In [ ]:
mlflow.end_run()